# Sesión 19 — Modelos de Difusión y Modelos de Lenguaje Grandes
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo V · Arquitecturas Avanzadas y AI Generativa**

## Objetivos de aprendizaje

1. Derivar los procesos hacia adelante (ruido) y hacia atrás (eliminación de ruido) de DDPM.
2. Comprender el score matching y su conexión con los autoencoders de eliminación de ruido.
3. Implementar un modelo de difusión 1-D mínimo para síntesis de ECG.
4. Comprender la arquitectura autoregresiva de GPT y en qué difiere de BERT.
5. Revisar aplicaciones clínicas de LLM y comprender cuantitativamente los riesgos de alucinación.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Ho, J., Jain, A. & Abbeel, P. (2020). Denoising diffusion probabilistic models. *NeurIPS*. |
| ★★★ | Song, Y. & Ermon, S. (2019). Generative modeling by estimating gradients of the data distribution. *NeurIPS*. (Score matching) |
| ★★☆ | Radford, A. et al. (2019). Language models are unsupervised multitask learners. OpenAI. (GPT-2) |
| ★★☆ | Wornow, M. et al. (2023). The shaky foundations of large language models for healthcare. *npj Digital Medicine*, 6. |
| ★★☆ | Alcaraz, J.M.L. & Strodthoff, N. (2023). Diffusion-based conditional ECG generation. *Computers in Biology and Medicine*. |
| ★☆☆ | Blog de difusión de Lilian Weng: https://lilianweng.github.io/posts/2021-07-11-diffusion-models/ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

rng    = np.random.default_rng(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
print(f'Dispositivo: {device}')

## Parte 1 — DDPM: el proceso de adición de ruido hacia adelante

El proceso hacia adelante destruye gradualmente una muestra de datos $\mathbf{x}_0$
añadiendo ruido gaussiano a lo largo de $T$ pasos:

$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_t;\; \sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\; \beta_t\mathbf{I})$$

Una identidad clave permite muestrear $\mathbf{x}_t$ directamente desde $\mathbf{x}_0$ en un solo paso:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_t;\; \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$$

donde $\alpha_t = 1-\beta_t$ y $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.

Así, en tiempo de inferencia: $\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon}$, con $\boldsymbol{\epsilon}\sim\mathcal{N}(\mathbf{0},\mathbf{I})$.

In [ ]:
# ── Programación de ruido DDPM ─────────────────────────────────────────────────
T_diff = 300   # pasos de tiempo de difusión

# Programación beta lineal (Ho et al. 2020)
beta_start, beta_end = 1e-4, 0.02
betas      = torch.linspace(beta_start, beta_end, T_diff)
alphas     = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)   # ᾱ_t

# Visualizar la programación de ruido
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
t_axis = np.arange(1, T_diff+1)
axes[0].plot(t_axis, betas.numpy(),      lw=2, color='tomato')
axes[0].set(xlabel='Paso de tiempo t', ylabel='β_t', title='Programación de ruido β_t')
axes[1].plot(t_axis, alpha_bars.numpy(), lw=2, color='steelblue')
axes[1].set(xlabel='Paso de tiempo t', ylabel='ᾱ_t',
            title='Retención de señal ᾱ_t\n(→0 significa ruido puro)')
axes[2].plot(t_axis, np.sqrt(alpha_bars.numpy()),       lw=2, label='√ᾱ_t  (escala de señal)')
axes[2].plot(t_axis, np.sqrt(1-alpha_bars.numpy()),     lw=2, label='√(1-ᾱ_t)  (escala de ruido)')
axes[2].set(xlabel='Paso de tiempo t', title='Contribución de señal vs ruido')
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

# ── Visualizar la adición de ruido hacia adelante en un latido de ECG ─────────
def crear_ecg(length=250, rng_=None):
    rng_ = rng_ or np.random.default_rng()
    t = np.arange(length)/250
    return (0.15*np.exp(-((t-0.12)**2)/(2*0.015**2)) +
            1.00*np.exp(-((t-0.22)**2)/(2*0.006**2)) +
           -0.12*np.exp(-((t-0.24)**2)/(2*0.008**2)) +
            0.25*np.exp(-((t-0.38)**2)/(2*0.025**2)) +
            rng_.normal(0, 0.02, length)).astype(np.float32)

x0 = torch.tensor(crear_ecg(rng_=rng)).unsqueeze(0)   # (1, 250)
x0 = (x0 - x0.mean()) / (x0.std() + 1e-8)             # normalizar

def q_sample(x0, t, alpha_bars):
    """Muestrea x_t desde x_0 en un solo paso (reparametrización)."""
    ab = alpha_bars[t].sqrt()
    noise = torch.randn_like(x0)
    return ab * x0 + (1 - alpha_bars[t]).sqrt() * noise, noise

timesteps_mostrar = [0, 30, 60, 120, 200, 299]
fig, axes = plt.subplots(len(timesteps_mostrar), 1, figsize=(13, 9), sharex=True)
t_ax = np.linspace(0, 1, 250)

for ax, t_show in zip(axes, timesteps_mostrar):
    xt, _ = q_sample(x0, t_show, alpha_bars)
    snr = alpha_bars[t_show].item() / (1 - alpha_bars[t_show].item() + 1e-8)
    ax.plot(t_ax, xt.squeeze().numpy(), lw=1.2, color='steelblue')
    ax.set(ylabel=f't={t_show}', yticks=[])
    ax.set_title(f't = {t_show}   SNR = {snr:.3f}   ᾱ_t = {alpha_bars[t_show]:.3f}',
                  fontsize=9, pad=2)

axes[-1].set_xlabel('Tiempo (s)')
fig.suptitle('Proceso hacia adelante de DDPM — latido de ECG progresivamente destruido por ruido gaussiano',
              y=1.01)
plt.tight_layout()
plt.show()

## Parte 2 — Proceso hacia atrás: aprender a eliminar el ruido

El modelo aprende a predecir el ruido $\boldsymbol{\epsilon}$ añadido en cada paso:

$$\mathcal{L} = \mathbb{E}_{t,\mathbf{x}_0,\boldsymbol{\epsilon}}\left[\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\|^2\right]$$

En tiempo de muestreo, aplicamos iterativamente:

$$\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t)\right) + \sigma_t\mathbf{z}, \quad \mathbf{z}\sim\mathcal{N}(\mathbf{0},\mathbf{I})$$

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Codifica el paso de tiempo escalar t en un vector de d dimensiones."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -np.log(10000) * torch.arange(half, device=t.device).float() / (half - 1)
        )
        args  = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)   # (B, dim)


class DenoiseMLP(nn.Module):
    """
    Red mínima de predicción de ruido para señales 1-D.
    ε_θ(x_t, t) — toma la señal ruidosa + el paso de tiempo, predice el ruido añadido.
    """
    def __init__(self, signal_len=250, t_emb_dim=32, hidden=256):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbedding(t_emb_dim)
        self.net = nn.Sequential(
            nn.Linear(signal_len + t_emb_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden),                  nn.SiLU(),
            nn.Linear(hidden, hidden),                  nn.SiLU(),
            nn.Linear(hidden, signal_len),
        )

    def forward(self, x, t):
        t_emb = self.time_emb(t)              # (B, t_emb_dim)
        inp   = torch.cat([x, t_emb], dim=1)  # (B, signal_len + t_emb_dim)
        return self.net(inp)


# ── Dataset: 1200 latidos de ECG normalizados ─────────────────────────────────
beats_np = np.array([crear_ecg(rng_=rng) for _ in range(1200)], dtype=np.float32)
beats_np = (beats_np - beats_np.mean(1, keepdims=True)) / (beats_np.std(1, keepdims=True)+1e-8)
beats_t  = torch.tensor(beats_np)
loader_diff = DataLoader(TensorDataset(beats_t), batch_size=64, shuffle=True)

# Mover los tensores de la programación al dispositivo
betas_d      = betas.to(device)
alphas_d     = alphas.to(device)
alpha_bars_d = alpha_bars.to(device)

# ── Entrenamiento ───────────────────────────────────────────────────────────────
denoiser = DenoiseMLP(signal_len=250, t_emb_dim=32, hidden=256).to(device)
opt_diff  = optim.Adam(denoiser.parameters(), lr=3e-4)

n_epochs_diff = 60
hist_diff = []

for ep in range(n_epochs_diff):
    denoiser.train()
    ep_loss = 0
    for (Xb,) in loader_diff:
        Xb = Xb.to(device)
        B  = Xb.size(0)

        # Muestrear pasos de tiempo aleatorios
        t_rand = torch.randint(0, T_diff, (B,), device=device)

        # Proceso hacia adelante: añadir ruido
        ab   = alpha_bars_d[t_rand].unsqueeze(1)       # (B, 1)
        eps  = torch.randn_like(Xb)
        Xt   = ab.sqrt() * Xb + (1-ab).sqrt() * eps   # x_t

        # Predecir el ruido
        eps_pred = denoiser(Xt, t_rand)
        loss     = F.mse_loss(eps_pred, eps)

        opt_diff.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(denoiser.parameters(), 1.0)
        opt_diff.step()
        ep_loss += loss.item()

    hist_diff.append(ep_loss / len(loader_diff))
    if (ep+1) % 15 == 0:
        print(f'Época {ep+1:3d}  pérdida = {hist_diff[-1]:.5f}')

plt.figure(figsize=(8, 3))
plt.plot(hist_diff, lw=2, color='steelblue')
plt.xlabel('Época')
plt.ylabel('Pérdida MSE de predicción de ruido')
plt.title('Pérdida de entrenamiento del modelo de difusión')
plt.tight_layout()
plt.show()

In [ ]:
# ── Muestreo DDPM (proceso hacia atrás) ───────────────────────────────────────
@torch.no_grad()
def ddpm_sample(model, n_samples, signal_len, T, betas, alphas, alpha_bars, device):
    model.eval()
    x = torch.randn(n_samples, signal_len, device=device)   # comenzar desde ruido puro
    trayectoria = [x.cpu().numpy().copy()]

    for t in reversed(range(T)):
        t_batch  = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)

        # Paso hacia atrás de DDPM
        alpha_t  = alphas[t]
        alpha_bt = alpha_bars[t]
        beta_t   = betas[t]

        x = (1/alpha_t.sqrt()) * (x - beta_t / (1-alpha_bt).sqrt() * eps_pred)
        if t > 0:
            sigma_t = beta_t.sqrt()
            x = x + sigma_t * torch.randn_like(x)

        if t % 50 == 0:
            trayectoria.append(x.cpu().numpy().copy())

    return x.cpu().numpy(), trayectoria


samples, traj = ddpm_sample(denoiser, n_samples=6, signal_len=250,
                             T=T_diff, betas=betas_d, alphas=alphas_d,
                             alpha_bars=alpha_bars_d, device=device)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
t_ax = np.linspace(0, 1, 250)

# Muestras generadas
for i in range(6):
    axes[0].plot(t_ax, samples[i] + i*3, lw=1.5,
                  color=plt.cm.Blues(0.4 + i*0.1))
axes[0].set(yticks=[], title='Latidos de ECG generados por DDPM (difusión inversa desde ruido puro)')

# Trayectoria de eliminación de ruido para una muestra
n_traj = len(traj)
t_labels = list(reversed(range(0, T_diff, 50))) + [0]
for k, snap in enumerate(traj[::-1][:6]):
    axes[1].plot(t_ax, snap[0] + k*3, lw=1.2,
                  color=plt.cm.Reds(0.3 + k*0.12),
                  label=f't={t_labels[k] if k < len(t_labels) else 0}')
axes[1].set(yticks=[], xlabel='Tiempo (s)',
            title='Trayectoria de eliminación de ruido — una muestra (ruido → señal)')
axes[1].legend(fontsize=8, loc='upper right', ncol=3)

plt.tight_layout()
plt.show()

## Parte 3 — Arquitectura GPT y generación autoregresiva

GPT usa un **Transformer decoder-only** con enmascaramiento causal (izquierda a derecha):

| | BERT (encoder) | GPT (decoder) |
|---|---|---|
| Atención | Bidireccional | Causal (izq. a der.) |
| Preentrenamiento | LM Enmascarado | Predicción del siguiente token |
| Caso de uso | Clasificación, NER | Generación, completado |
| Contexto | Secuencia completa | Solo tokens pasados |

El objetivo de entrenamiento es la simple predicción del siguiente token:
$$\mathcal{L} = -\sum_t \log p_\theta(x_t \mid x_1, \ldots, x_{t-1})$$

In [ ]:
# ── GPT mínimo para secuencias de tokens de ECG ───────────────────────────────
# Discretizar amplitudes de ECG en 64 bins → tratarlos como vocabulario de tokens

VOCAB_SIZE = 64
SEQ_LEN    = 250

def cuantizar(signal, n_bins=VOCAB_SIZE):
    """Mapea la señal continua a tokens enteros."""
    mn, mx = signal.min(), signal.max()
    return np.clip(((signal - mn) / (mx - mn + 1e-8) * n_bins).astype(int), 0, n_bins-1)

tokens_np = np.array([cuantizar(b) for b in beats_np], dtype=np.int64)  # (1200, 250)
tokens_t  = torch.tensor(tokens_np)

class TinyGPT(nn.Module):
    """
    GPT mínimo: decoder Transformer causal para secuencias de tokens discretos.
    Predice el siguiente token en cada posición simultáneamente durante el entrenamiento.
    """
    def __init__(self, vocab_size, seq_len, d_model=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(seq_len,    d_model)
        self.dropout   = nn.Dropout(dropout)

        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, vocab_size)

        # Máscara causal: la posición i solo puede atender a posiciones ≤ i
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        self.register_buffer('causal_mask', mask)

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        emb  = self.dropout(self.token_emb(x) + self.pos_emb(pos))   # (B, T, d)
        out  = self.transformer(emb, mask=self.causal_mask[:T,:T])    # (B, T, d)
        return self.head(out)    # (B, T, vocab_size) — logits en cada posición

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0):
        self.eval()
        x = prompt.clone()
        for _ in range(max_new_tokens):
            logits = self(x[:, -SEQ_LEN:])[:, -1, :] / temperature
            probs  = F.softmax(logits, dim=-1)
            nxt    = torch.multinomial(probs, 1)
            x      = torch.cat([x, nxt], dim=1)
        return x


gpt = TinyGPT(VOCAB_SIZE, SEQ_LEN).to(device)
print(f'TinyGPT: {sum(p.numel() for p in gpt.parameters()):,} parámetros')

# Entrenar
loader_gpt = DataLoader(TensorDataset(tokens_t), batch_size=32, shuffle=True)
opt_gpt    = optim.AdamW(gpt.parameters(), lr=3e-4, weight_decay=1e-2)
hist_gpt   = []

for ep in range(30):
    gpt.train()
    ep_loss = 0
    for (seqs,) in loader_gpt:
        seqs = seqs.to(device)
        # Entrada: todos menos el último token; objetivo: todos menos el primero
        logits = gpt(seqs[:, :-1])              # (B, T-1, V)
        loss   = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            seqs[:, 1:].reshape(-1)
        )
        opt_gpt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        opt_gpt.step()
        ep_loss += loss.item()
    hist_gpt.append(ep_loss / len(loader_gpt))
    if (ep+1) % 10 == 0:
        print(f'Época {ep+1:2d}  pérdida={hist_gpt[-1]:.4f}')

# Generar un nuevo latido de forma autoregresiva
prompt = torch.tensor([[VOCAB_SIZE//2]], device=device)   # token de inicio
gen_tokens = gpt.generate(prompt, max_new_tokens=249, temperature=0.8)
gen_signal  = gen_tokens[0].cpu().numpy().astype(float)
gen_signal  = (gen_signal - gen_signal.mean()) / (gen_signal.std() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_gpt, lw=2)
axes[0].set(xlabel='Época', ylabel='Entropía cruzada', title='Pérdida de entrenamiento de TinyGPT')
axes[1].plot(np.linspace(0,1,250), gen_signal, lw=1.5, color='tomato', label='Generado por GPT')
axes[1].plot(np.linspace(0,1,250), beats_np[0], lw=1.5, color='steelblue', alpha=0.5, label='Latido real')
axes[1].set(xlabel='Tiempo (s)', yticks=[], title='Generación autoregresiva de ECG (TinyGPT)')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

## Parte 4 — LLM clínicos: capacidades, riesgos y auditoría de alucinaciones

Esta sección es principalmente expositiva. El código a continuación implementa una
**auditoría de alucinaciones** estructurada — una herramienta práctica para evaluar
cualquier LLM clínico.

In [ ]:
import pandas as pd

# Resultados simulados de auditoría de alucinaciones (reemplazar con salidas reales de LLM)
# Cada entrada: una pregunta clínica, la respuesta del LLM, y la anotación experta
datos_auditoria = [
    {'pregunta': '¿Tratamiento de primera línea para STEMI?',
     'respuesta_llm': 'ICP primaria dentro de los 90 minutos del primer contacto médico.',
     'correcto': True, 'tipo_error': None},
    {'pregunta': '¿Umbral de troponina para NSTEMI (hs-cTnI)?',
     'respuesta_llm': '52 ng/L para hombres, 16 ng/L para mujeres (percentil 99).',
     'correcto': True, 'tipo_error': None},
    {'pregunta': '¿Contraindicación para trombólisis?',
     'respuesta_llm': 'Cirugía reciente es contraindicación absoluta dentro de 6 meses.',
     'correcto': False, 'tipo_error': 'Error de umbral (debería ser 3 meses para cirugía mayor)'},
    {'pregunta': '¿Fármaco de elección para control de frecuencia en FA con ICFEp?',
     'respuesta_llm': 'Se prefieren betabloqueadores o BCC no-dihidropiridínicos.',
     'correcto': True, 'tipo_error': None},
    {'pregunta': '¿INR objetivo para válvula mitral mecánica?',
     'respuesta_llm': 'El INR objetivo es 2.0–3.0.',
     'correcto': False, 'tipo_error': 'Error factual (rango correcto 2.5–3.5 para mitral)'},
    {'pregunta': '¿Hallazgo de EEG diagnóstico de epilepsia de ausencia?',
     'respuesta_llm': 'Descargas generalizadas de punta-onda a 3 Hz.',
     'correcto': True, 'tipo_error': None},
    {'pregunta': '¿FAE de primera línea para epilepsia mioclónica juvenil?',
     'respuesta_llm': 'La carbamazepina suele ser la primera opción.',
     'correcto': False, 'tipo_error': 'Error factual (carbamazepina empeora la EMJ; correcto: valproato/levetiracetam)'},
    {'pregunta': '¿Límite superior normal de QTc (hombres)?',
     'respuesta_llm': 'QTc > 440 ms se considera prolongado en hombres.',
     'correcto': True, 'tipo_error': None},
    {'pregunta': '¿Criterios de Duke para endocarditis infecciosa?',
     'respuesta_llm': 'El diagnóstico requiere 2 mayores, 1 mayor + 3 menores, o 5 menores.',
     'correcto': True, 'tipo_error': None},
    {'pregunta': '¿Antídoto para sobredosis de heparina?',
     'respuesta_llm': 'La vitamina K revierte la anticoagulación por heparina en horas.',
     'correcto': False, 'tipo_error': 'Confabulación (vitamina K revierte warfarina; heparina = sulfato de protamina)'},
]

df_auditoria = pd.DataFrame(datos_auditoria)
n_correcto  = df_auditoria['correcto'].sum()
n_total     = len(df_auditoria)
tipos_error = df_auditoria[~df_auditoria['correcto']]['tipo_error'].tolist()

print('=== Auditoría de Alucinaciones de LLM Clínico ===')
print(f'Exactitud: {n_correcto}/{n_total} ({100*n_correcto/n_total:.0f}%)')
print(f'\nErrores identificados ({n_total - n_correcto}):')
for i, err in enumerate(tipos_error, 1):
    print(f'  {i}. {err}')

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barra de exactitud
axes[0].bar(['Correcto', 'Alucinación/Error'],
             [n_correcto, n_total - n_correcto],
             color=['steelblue', 'tomato'], edgecolor='white', width=0.4)
axes[0].set(ylabel='Conteo', title=f'Exactitud clínica del LLM\n({n_correcto}/{n_total} correctas)')
for i, v in enumerate([n_correcto, n_total - n_correcto]):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Desglose de tipos de error
categorias_error = {'Error factual': 0, 'Error de umbral': 0, 'Confabulación': 0}
for e in tipos_error:
    for cat in categorias_error:
        if cat.lower() in e.lower():
            categorias_error[cat] += 1
            break

axes[1].pie(list(categorias_error.values()),
             labels=list(categorias_error.keys()),
             colors=['tomato', 'darkorange', 'gold'],
             autopct='%1.0f%%', startangle=90)
axes[1].set_title('Desglose por tipo de error')

plt.suptitle('Auditoría de alucinaciones de LLM clínico — resultados simulados\n'
             'En la práctica: siempre anclar las salidas en bases de conocimiento clínico curadas',
             y=1.02)
plt.tight_layout()
plt.show()

print('\nEstrategias de mitigación:')
for s in [
    '1. Generación Aumentada por Recuperación (RAG) contra guías clínicas curadas',
    '2. Ajuste fino en preguntas y respuestas médicas verificadas con RLHF de retroalimentación clínica',
    '3. Puntuaciones de confianza obligatorias — negarse a responder cuando hay incertidumbre',
    '4. Humano en el ciclo (human-in-the-loop) para decisiones de alto riesgo (nunca totalmente autónomo)',
    '5. Re-auditoría regular conforme cambian las guías (deriva temporal de conceptos)',
]:
    print(f'  {s}')

## ✏️ Ejercicios

1. **Programación de ruido coseno.** Implementa la programación coseno de Nichol &
   Dhariwal (2021): $\bar{\alpha}_t = \cos^2\!\left(\frac{t/T + s}{1+s}\cdot\frac{\pi}{2}\right)$.
   Compara la trayectoria de adición de ruido hacia adelante con la programación lineal.
   ¿En qué paso de tiempo la señal se vuelve indistinguible del ruido bajo cada programación?

2. **Difusión condicional.** Extiende el `DenoiseMLP` para aceptar una etiqueta de clase
   (N vs PVC) como embedding adicional. Entrena un modelo de difusión condicional y
   genera latidos de ECG específicos por clase. Evalúa la calidad entrenando un
   clasificador solo en datos generados y probándolo en datos reales (benchmark
   train-on-synthetic, test-on-real).

3. **Muestreo DDIM.** Implementa DDIM (Denoising Diffusion Implicit Models — Song et al.
   2021), que permite muestrear en tan solo 10-50 pasos en lugar de 300. Compara la
   calidad de las muestras (FID en características del latido) vs el número de pasos
   de muestreo.

4. **Barrido de temperatura en GPT.** Genera 50 latidos de ECG a temperaturas
   ∈ {0.5, 0.8, 1.0, 1.5, 2.0}. Para cada temperatura, calcula la media y varianza de
   las señales generadas. Grafica diversidad vs fidelidad. ¿Qué temperatura produce
   el mejor equilibrio?

5. *(Desafío)* **RAG para LLM clínicos.** Construye un pipeline RAG simple:
   (a) codifica oraciones de una guía clínica (ej. guías ESC de insuficiencia cardíaca,
   disponibles libremente como PDF) usando un sentence-transformer, (b) almacena en un
   índice FAISS, (c) para cada pregunta de la auditoría anterior, recupera los 3 pasajes
   más relevantes, (d) anteponlos al prompt del LLM. Mide la mejora en exactitud frente
   a la línea base.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| MIT-BIH Arrhythmia | https://physionet.org/content/mitdb/ | ECG para modelo de difusión |
| MedQA (USMLE) | https://github.com/jind11/MedQA | Benchmark de QA médico |
| Guías ESC | https://www.escardio.org/Guidelines | PDF libre para pipeline RAG |
| PubMedQA | https://pubmedqa.github.io | QA biomédico sí/no |